# Chapter 8: Introduction to Linear Regression
**Data Mining Course — Haydar Kılıç**

---

In this notebook we will cover:
1. Fitting a Line, Residuals, and Correlation
2. Fitting a Line Using Least Squares Regression
3. Types of Outliers in Linear Regression
4. Inference for Linear Regression

In [ ]:
# Required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.unicode_minus'] = False

print("Libraries loaded successfully!")

---
## SECTION 1: Fitting a Line, Residuals, and Correlation

### 1.1 Dataset: Poverty — High School Graduation Rate

We work with data from 50 US states + DC.  
- **Response variable (y):** Poverty rate (%)  
- **Explanatory variable (x):** High school graduation rate (%)

In [ ]:
# Synthetic data based on summary statistics from the slides
# x_bar=86.01, sx=3.73, y_bar=11.35, sy=3.1, R=-0.75
np.random.seed(42)
n = 51  # 50 states + DC

x_bar, sx = 86.01, 3.73
y_bar, sy = 11.35, 3.10
R = -0.75

# Generate bivariate normal distribution from covariance matrix
cov_xy     = R * sx * sy
cov_matrix = [[sx**2, cov_xy], [cov_xy, sy**2]]
data = np.random.multivariate_normal([x_bar, y_bar], cov_matrix, n)

hs_grad  = data[:, 0]
poverty  = data[:, 1]

# Clip to realistic ranges
hs_grad = np.clip(hs_grad, 78, 93)
poverty = np.clip(poverty, 5.5, 19)

# Special points representing DC and RI
# DC: hs_grad=86, poverty=16.8  (residual ≈ +5.44)
# RI: hs_grad=81, poverty=10.3  (residual ≈ -4.16)
hs_grad[-2], poverty[-2] = 86.0, 16.8   # DC
hs_grad[-1], poverty[-1] = 81.0, 10.3   # RI

df = pd.DataFrame({'hs_grad': hs_grad, 'poverty': poverty})

print("Data summary:")
print(df.describe().round(2))
print(f"\nCorrelation: {df['hs_grad'].corr(df['poverty']):.3f}")

In [ ]:
# Scatter plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(hs_grad, poverty, color='steelblue', alpha=0.7, s=60, edgecolors='white')
ax.set_xlabel('High School Graduation Rate (%)')
ax.set_ylabel('Poverty Rate (%)')
ax.set_title('Relationship Between High School Graduation Rate and Poverty Rate\n(US — 50 States + DC, 2012)')

# Label DC and RI
ax.annotate('DC', xy=(86, 16.8), xytext=(84.5, 17.8),
            fontsize=10, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred', lw=1.2))
ax.annotate('RI', xy=(81, 10.3), xytext=(79.5, 9.2),
            fontsize=10, color='darkgreen',
            arrowprops=dict(arrowstyle='->', color='darkgreen', lw=1.2))

plt.tight_layout()
plt.show()

print("Observation:")
print("  - Relationship: linear, negative, moderately strong")
print("  - As the high school graduation rate increases, the poverty rate decreases.")

---
### 1.2 Regression Model and Prediction

Model from the slides:
$$\widehat{\text{poverty}} = 64{.}78 - 0{.}62 \times \text{hs\_grad}$$

The hat (^) symbol indicates that this is a **prediction**.

In [ ]:
# Coefficients from the slides
b0 = 64.78
b1 = -0.62

def predict(hs_grad_value):
    """Predict poverty rate using the linear model"""
    return b0 + b1 * hs_grad_value

# Georgia example (from slides): hs_grad = 85.1%
georgia_hs    = 85.1
georgia_pred  = predict(georgia_hs)
print(f"Prediction for Georgia (hs grad = {georgia_hs}%):")
print(f"  ŷ = {b0} + ({b1}) × {georgia_hs} = {georgia_pred:.3f}")
print()

# Prediction table for several states
states = {
    'Georgia':     85.1,
    'Mississippi': 80.5,
    'Minnesota':   91.8,
    'Utah':        90.4,
    'DC':          86.0,
    'RI':          81.0
}

print(f"{'State':<12} {'HS Grad %':>12} {'Prediction (ŷ)':>14}")
print("-" * 40)
for state, hs in states.items():
    print(f"{state:<12} {hs:>12.1f} {predict(hs):>14.2f}")

---
### 1.3 Residuals

A **residual** is the difference between the observed $(y_i)$ and the predicted $(\hat{y}_i)$ value:
$$e_i = y_i - \hat{y}_i$$

Fundamental equation:
$$\text{Data} = \text{Fit} + \text{Residual}$$

In [ ]:
# Compute predicted values and residuals for all data
y_hat     = predict(hs_grad)
residuals = poverty - y_hat

# Compute DC and RI residuals
dc_hs, dc_pov = 86.0, 16.8
ri_hs, ri_pov = 81.0, 10.3

dc_yhat   = predict(dc_hs)
ri_yhat   = predict(ri_hs)
dc_resid  = dc_pov - dc_yhat
ri_resid  = ri_pov - ri_yhat

print("DC (Washington D.C.):")
print(f"  Observed:  {dc_pov}")
print(f"  Predicted: {b0} - {abs(b1)} × {dc_hs} = {dc_yhat:.2f}")
print(f"  Residual:  {dc_pov} - {dc_yhat:.2f} = {dc_resid:.2f}  ← {abs(dc_resid):.2f} MORE than predicted")
print()
print("RI (Rhode Island):")
print(f"  Observed:  {ri_pov}")
print(f"  Predicted: {b0} - {abs(b1)} × {ri_hs} = {ri_yhat:.2f}")
print(f"  Residual:  {ri_pov} - {ri_yhat:.2f} = {ri_resid:.2f}  ← {abs(ri_resid):.2f} LESS than predicted")

In [ ]:
# Visualize residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_line = np.linspace(77, 94, 200)
y_line = predict(x_line)

# Left: scatter + regression line + residual lines
ax = axes[0]
ax.scatter(hs_grad[:-2], poverty[:-2], color='steelblue', alpha=0.6, s=50, zorder=3)
ax.plot(x_line, y_line, 'r-', linewidth=2.5, label='Regression line', zorder=2)

# Residual lines
for i in range(len(hs_grad)-2):
    ax.plot([hs_grad[i], hs_grad[i]], [poverty[i], y_hat[i]],
            color='olive', alpha=0.5, linewidth=1, zorder=1)

# DC and RI
ax.scatter([dc_hs, ri_hs], [dc_pov, ri_pov],
           color=['darkred', 'darkgreen'], s=80, zorder=5)
ax.plot([dc_hs, dc_hs], [dc_yhat, dc_pov], 'r-', linewidth=2)
ax.plot([ri_hs, ri_hs], [ri_yhat, ri_pov], 'g-', linewidth=2)
ax.annotate(f'DC\n+{dc_resid:.2f}', xy=(dc_hs+0.2, (dc_pov+dc_yhat)/2),
            color='darkred', fontsize=9, fontweight='bold')
ax.annotate(f'RI\n{ri_resid:.2f}', xy=(ri_hs-2.2, (ri_pov+ri_yhat)/2),
            color='darkgreen', fontsize=9, fontweight='bold')

ax.set_xlabel('High School Graduation Rate (%)')
ax.set_ylabel('Poverty Rate (%)')
ax.set_title('Regression Line and Residuals')
ax.legend()

# Right: residual plot
ax2 = axes[1]
ax2.scatter(hs_grad[:-2], residuals[:-2], color='steelblue', alpha=0.6, s=50)
ax2.scatter([dc_hs, ri_hs], [dc_resid, ri_resid],
            color=['darkred', 'darkgreen'], s=80, zorder=5)
ax2.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax2.set_xlabel('High School Graduation Rate (%)')
ax2.set_ylabel('Residual (Observed − Predicted)')
ax2.set_title('Residual Plot')
ax2.annotate('DC', xy=(dc_hs+0.2, dc_resid), color='darkred', fontweight='bold')
ax2.annotate('RI', xy=(ri_hs+0.2, ri_resid), color='darkgreen', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nMean of residuals (should ≈ 0): {residuals.mean():.4f}")
print(f"Standard deviation of residuals: {residuals.std():.4f}")

---
### 1.4 Correlation

**Correlation** describes the **strength of the linear relationship** between two variables.  
- Takes values between $-1$ (perfect negative) and $+1$ (perfect positive)  
- A value of $0$ indicates no linear relationship

> ⚠️ Correlation only measures **linear** relationships!

In [ ]:
# Visualize different correlation values
np.random.seed(0)
n_demo = 80

correlations = {
    'R ≈ +0.95 (Strong Positive)':  0.95,
    'R ≈ +0.50 (Moderate Positive)': 0.50,
    'R ≈ 0.00 (No Relationship)':   0.00,
    'R ≈ -0.75 (Moderate Negative)':-0.75,
    'R ≈ -0.95 (Strong Negative)':  -0.95,
    'R = ? (Non-linear)':           None
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (title, r) in enumerate(correlations.items()):
    ax = axes[idx]
    if r is not None:
        cov  = [[1, r], [r, 1]]
        xy   = np.random.multivariate_normal([0, 0], cov, n_demo)
        x_d, y_d = xy[:, 0], xy[:, 1]
        r_calc   = np.corrcoef(x_d, y_d)[0, 1]
        label    = f'R = {r_calc:.2f}'
    else:
        # Non-linear: U-shape
        x_d    = np.linspace(-3, 3, n_demo) + np.random.normal(0, 0.3, n_demo)
        y_d    = x_d**2 + np.random.normal(0, 0.5, n_demo)
        r_calc = np.corrcoef(x_d, y_d)[0, 1]
        label  = f'R = {r_calc:.2f} (U-shape!)'

    ax.scatter(x_d, y_d, alpha=0.5, s=25, color='steelblue')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(label, fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle('Visualising Different Correlation Values', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("NOTE: Bottom-right (U-shape) — despite a strong relationship between the variables,")
print("the correlation ≈ 0. Correlation only measures LINEAR relationships!")

---
## SECTION 2: Fitting a Line Using Least Squares Regression

### 2.1 The Least Squares Method

**Goal:** Find the line that minimises the sum of squared residuals:
$$\text{Minimise: } e_1^2 + e_2^2 + \cdots + e_n^2 = \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

**Why squared?**
- The most widely used method
- Easier to compute by hand and with software
- Penalises large residuals more heavily

In [ ]:
# Least squares formulas
# b1 = (sy / sx) * R
# b0 = y_bar - b1 * x_bar

sx_calc    = np.std(hs_grad, ddof=1)
sy_calc    = np.std(poverty, ddof=1)
R_calc     = np.corrcoef(hs_grad, poverty)[0, 1]
x_bar_calc = np.mean(hs_grad)
y_bar_calc = np.mean(poverty)

b1_calc = (sy_calc / sx_calc) * R_calc
b0_calc = y_bar_calc - b1_calc * x_bar_calc

print("Summary Statistics:")
print(f"  x̄  (mean hs grad)      = {x_bar_calc:.2f}")
print(f"  ȳ  (mean poverty)      = {y_bar_calc:.2f}")
print(f"  sx (sd hs grad)        = {sx_calc:.2f}")
print(f"  sy (sd poverty)        = {sy_calc:.2f}")
print(f"  R  (correlation)       = {R_calc:.3f}")
print()
print("Slope Calculation:")
print(f"  b1 = (sy/sx) × R = ({sy_calc:.2f}/{sx_calc:.2f}) × ({R_calc:.3f}) = {b1_calc:.4f}")
print()
print("Intercept Calculation:")
print(f"  b0 = ȳ - b1×x̄ = {y_bar_calc:.2f} - ({b1_calc:.4f})×{x_bar_calc:.2f} = {b0_calc:.4f}")
print()
print(f"Model: ŷ = {b0_calc:.2f} + ({b1_calc:.2f}) × x")
print(f"Model from slides: ŷ = 64.78 - 0.62 × x  (matches ✓)")

In [ ]:
# Verification with sklearn
model = LinearRegression()
model.fit(hs_grad.reshape(-1, 1), poverty)

print("sklearn LinearRegression results:")
print(f"  b0 (intercept) = {model.intercept_:.4f}")
print(f"  b1 (slope)     = {model.coef_[0]:.4f}")
print(f"  R²             = {model.score(hs_grad.reshape(-1,1), poverty):.4f}")

---
### 2.2 Interpretation of Model Coefficients

$$\widehat{\text{poverty}} = 64{.}78 - 0{.}62 \times \text{hs\_grad}$$

- **Slope (b₁ = -0.62):** For each additional **1%** increase in the high school graduation rate, the poverty rate is expected to be **0.62 points** lower on average.
- **Intercept (b₀ = 64.78):** In a state where 0% graduated from high school (unrealistic — extrapolation!), the poverty rate would be expected to be 64.78%.

> ⚠️ These statements are **not causal** unless the study is a randomised controlled experiment!

In [ ]:
# Visualise the regression line and intercept
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Model within the data range
ax1 = axes[0]
x_range = np.linspace(77, 94, 300)
y_range = b0 + b1 * x_range
ax1.scatter(hs_grad, poverty, color='steelblue', alpha=0.6, s=55, zorder=3)
ax1.plot(x_range, y_range, 'r-', linewidth=2.5)

# Highlight the mean point — regression always passes through (x̄, ȳ)!
ax1.scatter([x_bar_calc], [y_bar_calc], color='orange', s=150, zorder=5,
            marker='*', label=f'(x̄={x_bar_calc:.1f}, ȳ={y_bar_calc:.1f})')
ax1.axvline(x_bar_calc, color='orange', linestyle=':', alpha=0.5)
ax1.axhline(y_bar_calc, color='orange', linestyle=':', alpha=0.5)
ax1.set_xlabel('High School Graduation Rate (%)')
ax1.set_ylabel('Poverty Rate (%)')
ax1.set_title('Regression Line (Data Range)')
ax1.legend()
ax1.text(88, 17, f'ŷ = {b0:.2f} + ({b1})×x', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Right: Full range — extrapolation danger
ax2 = axes[1]
x_full = np.linspace(0, 100, 300)
y_full = b0 + b1 * x_full
ax2.scatter(hs_grad, poverty, color='steelblue', alpha=0.6, s=55)
ax2.plot(x_full, y_full, 'r-', linewidth=2)
ax2.axvline(x=0, color='purple', linestyle='--', alpha=0.5)
ax2.scatter([0], [b0], color='purple', s=120, zorder=5, label=f'Intercept = {b0}')
ax2.fill_betweenx([-10, 80], 0, 75, alpha=0.05, color='red', label='Extrapolation region')
ax2.set_xlim(-2, 103)
ax2.set_ylim(-5, 72)
ax2.set_xlabel('High School Graduation Rate (%)')
ax2.set_ylabel('Poverty Rate (%)')
ax2.set_title('Full Range — Extrapolation Danger!')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("⚠️  Intercept (x=0): No state in the dataset has 0% high school graduation!")
print("    Therefore, interpreting the intercept is unreliable (extrapolation).")

---
### 2.3 Model Conditions

The least squares line requires **3 key conditions**:

| Condition | Diagnostic Method |
|---|---|
| 1. Linearity | Scatter plot or residual plot |
| 2. Nearly normal residuals | Histogram |
| 3. Constant variability (Homoscedasticity) | Residual plot |

In [ ]:
# Full diagnostic plots for the 3 conditions
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

y_pred_diag = predict(hs_grad)
resid_diag  = poverty - y_pred_diag

# ---- CONDITION 1: LINEARITY ----
# Good example
ax = axes[0, 0]
ax.scatter(hs_grad, poverty, color='steelblue', alpha=0.6, s=40)
ax.plot(np.linspace(77, 94, 100), predict(np.linspace(77, 94, 100)), 'r-', lw=2)
ax.set_title('Condition 1 — Linearity ✓\n(Good Example)', color='green', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('y')

# Bad example — curved relationship
ax = axes[1, 0]
x_bad = np.linspace(-3, 3, 60) + np.random.normal(0, 0.2, 60)
y_bad = x_bad**2 + np.random.normal(0, 0.3, 60)
m_bad = LinearRegression().fit(x_bad.reshape(-1,1), y_bad)
ax.scatter(x_bad, y_bad, color='steelblue', alpha=0.6, s=40)
x_s = np.linspace(-3.5, 3.5, 100)
ax.plot(x_s, m_bad.predict(x_s.reshape(-1,1)), 'r-', lw=2)
ax.set_title('Condition 1 — Linearity ✗\n(Curvature in residual plot)', color='red', fontsize=10)
ax.set_xlabel('x'); ax.set_ylabel('Residual')

# ---- CONDITION 2: NEARLY NORMAL RESIDUALS ----
ax = axes[0, 1]
ax.hist(resid_diag, bins=12, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--')
ax.set_title('Condition 2 — Nearly Normal Residuals ✓\n(Histogram)', color='green', fontsize=10)
ax.set_xlabel('Residual'); ax.set_ylabel('Frequency')

# Bad example — skewed distribution
ax = axes[1, 1]
skewed_res = np.random.exponential(scale=1.5, size=100) - 1.5
ax.hist(skewed_res, bins=12, color='salmon', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--')
ax.set_title('Condition 2 — Normal Residuals ✗\n(Skewed distribution)', color='red', fontsize=10)
ax.set_xlabel('Residual'); ax.set_ylabel('Frequency')

# ---- CONDITION 3: CONSTANT VARIABILITY ----
ax = axes[0, 2]
ax.scatter(y_pred_diag, resid_diag, color='steelblue', alpha=0.6, s=40)
ax.axhline(0, color='red', linestyle='--')
ax.set_title('Condition 3 — Constant Variability ✓\n(Homoscedasticity)', color='green', fontsize=10)
ax.set_xlabel('Predicted (ŷ)'); ax.set_ylabel('Residual')

# Bad example — fan shape
ax = axes[1, 2]
x_fan = np.linspace(1, 10, 80)
y_fan = 2 * x_fan + np.random.normal(0, x_fan * 0.5, 80)
m_fan = LinearRegression().fit(x_fan.reshape(-1,1), y_fan)
res_fan = y_fan - m_fan.predict(x_fan.reshape(-1,1))
ax.scatter(m_fan.predict(x_fan.reshape(-1,1)), res_fan, color='salmon', alpha=0.6, s=40)
ax.axhline(0, color='red', linestyle='--')
ax.set_title('Condition 3 — Constant Variability ✗\n(Fan shape — Heteroscedasticity)', color='red', fontsize=10)
ax.set_xlabel('Predicted (ŷ)'); ax.set_ylabel('Residual')

plt.suptitle('Diagnostic Plots for Least Squares Conditions\n(Green: condition met | Red: condition violated)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
### 2.4 R² — Coefficient of Determination

$$R^2 = r^2 = (-0{.}75)^2 = 0{.}56$$

**Interpretation:** **56%** of the variability in the response variable is explained by the model.

In [ ]:
# R² calculation — two methods
R_corr    = np.corrcoef(hs_grad, poverty)[0, 1]
R2_corr   = R_corr**2

SS_tot    = np.sum((poverty - np.mean(poverty))**2)
SS_res    = np.sum((poverty - y_pred_diag)**2)
R2_direct = 1 - SS_res / SS_tot

print("R² Calculation:")
print(f"  Method 1 — Squared correlation: R² = ({R_corr:.4f})² = {R2_corr:.4f}")
print(f"  Method 2 — SS formula:          R² = 1 - SS_res/SS_tot = {R2_direct:.4f}")
print()
print(f"  Value from slides: R² = (-0.75)² = 0.5625")
print()
print("Interpretation:")
print(f"  {R2_direct*100:.1f}% of the variability in poverty rates across 51 states")
print(f"  is explained by the model using high school graduation rate as the explanatory variable.")
print(f"  The remaining {(1-R2_direct)*100:.1f}% is explained by variables not included in the model or")
print(f"  by natural randomness in the data.")

In [ ]:
# R² visualisation: total variance vs explained variance
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Without the model (mean only)
ax1 = axes[0]
ax1.scatter(hs_grad, poverty, color='steelblue', alpha=0.5, s=40)
ax1.axhline(np.mean(poverty), color='gray', linestyle='--', linewidth=2,
            label=f'ȳ = {np.mean(poverty):.2f} (mean model)')
for i in range(len(hs_grad)):
    ax1.plot([hs_grad[i], hs_grad[i]], [poverty[i], np.mean(poverty)],
             color='gray', alpha=0.3, linewidth=0.8)
ax1.set_title(f'Total Variability (SS_tot)\nR² = 0 → nothing explained')
ax1.set_xlabel('High School Graduation Rate (%)'); ax1.set_ylabel('Poverty (%)')
ax1.legend(fontsize=9)

# Right: With the model (regression line)
ax2 = axes[1]
ax2.scatter(hs_grad, poverty, color='steelblue', alpha=0.5, s=40)
ax2.plot(x_range, predict(x_range), 'r-', linewidth=2, label='Regression line')
for i in range(len(hs_grad)):
    ax2.plot([hs_grad[i], hs_grad[i]], [poverty[i], y_pred_diag[i]],
             color='green', alpha=0.3, linewidth=0.8)
ax2.set_title(f'Unexplained Variability (SS_res)\nR² = {R2_direct:.2f} → {R2_direct*100:.0f}% explained')
ax2.set_xlabel('High School Graduation Rate (%)'); ax2.set_ylabel('Poverty (%)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## SECTION 3: Types of Outliers

In linear regression there are three types of special points:

| Term | Definition |
|---|---|
| **Outlier** | A point located far from the main cloud |
| **High-leverage point** | A point far from the cloud horizontally (in x) |
| **Influential point** | A high-leverage point that actually changes the slope of the regression line |

In [ ]:
# Compare types of outliers
np.random.seed(7)
n_out = 40

x_base = np.random.normal(5, 1, n_out)
y_base = 2 * x_base + np.random.normal(0, 1, n_out)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scenarios = [
    {
        'title':   'Outlier\n(Large Residual, Low Leverage)',
        'x_out':   5.0, 'y_out': 20.0,
        'color':   'red',
        'note':    'At centre of x-axis\nbut very high on y-axis'
    },
    {
        'title':   'High-Leverage + Influential\n(Dramatically changes slope)',
        'x_out':   13.0, 'y_out': 8.0,
        'color':   'darkred',
        'note':    'Far on x-axis\nAND deviates from the model'
    },
    {
        'title':   'High-Leverage + Non-influential\n(Follows the trend)',
        'x_out':   13.0, 'y_out': 26.0,
        'color':   'orange',
        'note':    'Far on x-axis\nBUT lies on the trend'
    }
]

for idx, scenario in enumerate(scenarios):
    ax = axes[idx]
    x_out_all = np.append(x_base, scenario['x_out'])
    y_out_all = np.append(y_base, scenario['y_out'])

    # Model without the outlier
    m_no  = LinearRegression().fit(x_base.reshape(-1,1), y_base)
    # Model with the outlier
    m_yes = LinearRegression().fit(x_out_all.reshape(-1,1), y_out_all)

    x_plt = np.linspace(min(x_out_all)-0.5, max(x_out_all)+0.5, 200)

    ax.scatter(x_base, y_base, color='steelblue', alpha=0.5, s=35, label='Normal points')
    ax.scatter([scenario['x_out']], [scenario['y_out']],
               color=scenario['color'], s=120, zorder=5,
               marker='D', label='Outlier point', edgecolors='black')

    ax.plot(x_plt, m_no.predict(x_plt.reshape(-1,1)),
            'g--', linewidth=2, label=f'Without outlier (b₁={m_no.coef_[0]:.2f})')
    ax.plot(x_plt, m_yes.predict(x_plt.reshape(-1,1)),
            'r-',  linewidth=2, label=f'With outlier (b₁={m_yes.coef_[0]:.2f})')

    slope_change = abs(m_yes.coef_[0] - m_no.coef_[0])
    ax.set_title(scenario['title'], fontsize=9)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=7.5)
    ax.text(0.05, 0.95, f"Slope change: {slope_change:.2f}",
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.suptitle('Effect of Different Outlier Types on the Regression Line',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Conclusion:")
print("  Scenario 1: Large residual but little effect on slope → 'outlier, low leverage'")
print("  Scenario 2: Both far and dramatically changes slope → 'influential point'")
print("  Scenario 3: Far but follows the trend → 'high-leverage but non-influential'")

---
## SECTION 4: Inference for Linear Regression

### 4.1 Hypothesis Test — Testing the Slope

**Hypotheses:**
$$H_0: \beta_1 = 0 \quad (\text{no relationship})$$
$$H_A: \beta_1 \neq 0 \quad (\text{a significant relationship exists})$$

**Test statistic:**
$$T = \frac{b_1 - 0}{SE_{b_1}}, \quad df = n - 2$$

In [ ]:
# Nature vs nurture (IQ) data from the slides
np.random.seed(15)
n_iq = 27

r_iq     = 0.882
bio_iq   = np.random.normal(100, 15, n_iq)
foster_noise = np.random.normal(0, np.sqrt(1 - r_iq**2) * 15, n_iq)
foster_iq    = r_iq * (bio_iq - 100) + 100 + foster_noise
foster_iq    = np.clip(foster_iq, 68, 135)

# Fit with sklearn
model_iq = LinearRegression()
model_iq.fit(bio_iq.reshape(-1, 1), foster_iq)

# Manual computation of standard error and t-statistic
y_pred_iq = model_iq.predict(bio_iq.reshape(-1, 1))
SS_res_iq = np.sum((foster_iq - y_pred_iq)**2)
df_iq     = n_iq - 2
MSE_iq    = SS_res_iq / df_iq
SE_b1_iq  = np.sqrt(MSE_iq / np.sum((bio_iq - bio_iq.mean())**2))
SE_b0_iq  = np.sqrt(MSE_iq * (1/n_iq + bio_iq.mean()**2 / np.sum((bio_iq - bio_iq.mean())**2)))

b1_iq = model_iq.coef_[0]
b0_iq = model_iq.intercept_

t_stat_b1  = b1_iq / SE_b1_iq
p_value_b1 = 2 * (1 - stats.t.cdf(abs(t_stat_b1), df=df_iq))

t_stat_b0  = b0_iq / SE_b0_iq
p_value_b0 = 2 * (1 - stats.t.cdf(abs(t_stat_b0), df=df_iq))

R2_iq = model_iq.score(bio_iq.reshape(-1, 1), foster_iq)

print("Regression Output — IQ Data (Nature vs Nurture?)")
print("="*65)
print(f"{'':15} {'Estimate':>10} {'Std Error':>10} {'t value':>10} {'p value':>10}")
print("-"*65)
print(f"{'(Intercept)':15} {b0_iq:>10.4f} {SE_b0_iq:>10.4f} {t_stat_b0:>10.2f} {p_value_b0:>10.4f}")
print(f"{'bioIQ':15} {b1_iq:>10.4f} {SE_b1_iq:>10.4f} {t_stat_b1:>10.2f} {p_value_b1:>10.6f}")
print("="*65)
print(f"Residual standard error: {np.sqrt(MSE_iq):.3f} on {df_iq} degrees of freedom")
print(f"R-squared: {R2_iq:.4f}")
print()
print(f"Values from slides:  b0≈9.21 (SE≈9.30), b1≈0.90 (SE≈0.096), t≈9.36")
print()
print(f"Conclusion: p-value ({'<0.0001' if p_value_b1 < 0.0001 else f'{p_value_b1:.4f}'}) << 0.05")
print(f"→ Reject H₀. Biological IQ is a significant predictor of IQ in the foster family.")

In [ ]:
# Visualise the test statistic on the t-distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: t-distribution and critical region
ax1 = axes[0]
t_range = np.linspace(-12, 12, 500)
t_pdf   = stats.t.pdf(t_range, df=df_iq)
t_crit  = stats.t.ppf(0.975, df=df_iq)

ax1.plot(t_range, t_pdf, 'b-', linewidth=2, label=f't-distribution (df={df_iq})')
ax1.fill_between(t_range, t_pdf, where=(t_range >= t_crit),  color='red', alpha=0.3, label=f'Critical region (α=0.05)')
ax1.fill_between(t_range, t_pdf, where=(t_range <= -t_crit), color='red', alpha=0.3)
ax1.axvline(t_stat_b1, color='darkred', linestyle='-', linewidth=2.5,
            label=f'T = {t_stat_b1:.2f} (observed)')
ax1.axvline(t_crit, color='orange', linestyle='--', linewidth=1.5,
            label=f't* = ±{t_crit:.2f}')
ax1.set_xlim(-12, 12)
ax1.set_xlabel('t value')
ax1.set_ylabel('Density')
ax1.set_title('Hypothesis Test: H₀: β₁ = 0')
ax1.legend(fontsize=8)
ax1.text(t_stat_b1 - 3.5, 0.15, f'p < 0.0001\nH₀ Rejected!',
         fontsize=10, color='darkred', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Right: Regression line
ax2 = axes[1]
ax2.scatter(bio_iq, foster_iq, color='steelblue', alpha=0.6, s=60)
x_iq_range = np.linspace(65, 140, 200)
ax2.plot(x_iq_range, model_iq.predict(x_iq_range.reshape(-1,1)),
         'r-', linewidth=2.5)
ax2.text(0.05, 0.92,
         f'ŷ = {b0_iq:.2f} + {b1_iq:.2f}×bioIQ\nR = {np.sqrt(R2_iq):.3f}, R² = {R2_iq:.4f}',
         transform=ax2.transAxes, fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax2.set_xlabel('Biological Twin IQ')
ax2.set_ylabel('Foster-raised Twin IQ')
ax2.set_title('Nature vs Nurture?\n(IQ — Identical Twins)')

plt.tight_layout()
plt.show()

---
### 4.2 Confidence Interval — For the Slope

$$b_1 \pm t^*_{df=n-2} \times SE_{b_1}$$

For the example from the slides (n=27, df=25):
$$0{.}9014 \pm 2{.}06 \times 0{.}0963 = (0{.}70 \ ; \ 1{.}10)$$

In [ ]:
# Compute and visualise confidence intervals
alpha_levels    = [0.10, 0.05, 0.01]
confidence_levels = [0.90, 0.95, 0.99]

print("Confidence Intervals for the Slope (β₁):")
print(f"  b₁ = {b1_iq:.4f}, SE = {SE_b1_iq:.4f}, df = {df_iq}")
print()
print(f"{'Confidence Level':>18} {'t* value':>12} {'Lower Bound':>12} {'Upper Bound':>12}")
print("-" * 58)
for alpha, conf in zip(alpha_levels, confidence_levels):
    t_crit_conf = stats.t.ppf(1 - alpha/2, df=df_iq)
    lower = b1_iq - t_crit_conf * SE_b1_iq
    upper = b1_iq + t_crit_conf * SE_b1_iq
    print(f"{conf*100:>17.0f}% {t_crit_conf:>12.3f} {lower:>12.4f} {upper:>12.4f}")

print()
t_star_25 = stats.t.ppf(0.975, df=25)
b1_slide  = 0.9014
se_slide  = 0.0963
lower_slide = b1_slide - t_star_25 * se_slide
upper_slide = b1_slide + t_star_25 * se_slide
print(f"Using slide values: {b1_slide} ± {t_star_25:.2f} × {se_slide}")
print(f"  95% CI: ({lower_slide:.4f} , {upper_slide:.4f})  ≈ (0.70 , 1.10) ✓")

In [ ]:
# Visualise confidence intervals as a bar chart
fig, ax = plt.subplots(figsize=(9, 4))

colors     = ['#2196F3', '#4CAF50', '#F44336']
y_positions = [0, 1, 2]

for i, (alpha, conf) in enumerate(zip(alpha_levels, confidence_levels)):
    t_c   = stats.t.ppf(1 - alpha/2, df=df_iq)
    lower = b1_iq - t_c * SE_b1_iq
    upper = b1_iq + t_c * SE_b1_iq
    ax.barh(y_positions[i], upper - lower, left=lower, height=0.4,
            color=colors[i], alpha=0.6, label=f'{conf*100:.0f}% CI')
    ax.scatter([b1_iq], [y_positions[i]], color='black', s=60, zorder=5)
    ax.text(upper + 0.01, y_positions[i], f'({lower:.3f}, {upper:.3f})',
            va='center', fontsize=9)

ax.axvline(0, color='red', linestyle='--', linewidth=1.5,
           label='β₁=0 (H₀)')
ax.set_yticks(y_positions)
ax.set_yticklabels(['90% CI', '95% CI', '99% CI'])
ax.set_xlabel('β₁ (Slope)')
ax.set_title('Confidence Intervals for the Slope\n(Black dot: point estimate)')
ax.legend()
plt.tight_layout()
plt.show()

print("\nNOTE: The H₀: β₁=0 line does not fall inside any confidence interval.")
print("→ H₀ is rejected at all confidence levels.")

---
## SECTION 5: Extra — Extrapolation

Applying a model's predictions to **values outside the range of the original data** is called **extrapolation**.

In [ ]:
# Extrapolation example: Marriage age time series
np.random.seed(20)
years_early = np.array([1900, 1910, 1920, 1930, 1940])
ages_early  = np.array([26.2, 26.1, 25.1, 24.6, 23.9])

model_marriage = LinearRegression()
model_marriage.fit(years_early.reshape(-1,1), ages_early)

years_recent = np.arange(1950, 1985, 2)
ages_recent  = 22.5 + 0.08 * (years_recent - 1960) + np.random.normal(0, 0.2, len(years_recent))

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(years_early,  ages_early,  color='steelblue', s=80, zorder=5,
           label='Early data (1900–1940)')
ax.scatter(years_recent, ages_recent, color='darkred',   s=50, zorder=5,
           label='Actual values (1950–1980)')

x_extrap = np.arange(1895, 1985, 1)
ax.plot(x_extrap, model_marriage.predict(x_extrap.reshape(-1,1)),
        'b-', linewidth=2, label="Model's extrapolation")
ax.axvspan(1945, 1985, alpha=0.1, color='red', label='Extrapolation region')
ax.axvline(1945, color='gray', linestyle='--', alpha=0.7)

ax.set_xlabel('Year')
ax.set_ylabel('Age at First Marriage (Men)')
ax.set_title('Danger of Extrapolation: Marriage Age\nModel based on 1900–1940 data poorly predicts post-1950 values')
ax.legend(fontsize=9)
ax.set_ylim(20, 28)
plt.tight_layout()
plt.show()

pred_1970 = model_marriage.predict([[1970]])[0]
print(f"Model prediction for 1970: {pred_1970:.1f} years")
print(f"Actual 1970 value: ~23.2 years")
print(f"Error: {abs(pred_1970 - 23.2):.1f} years — how misleading extrapolation can be!")

---
## SECTION 6: Summary and Checklist

What we learned in this lecture:

In [ ]:
# Complete linear regression analysis — all steps
print("=" * 60)
print("LINEAR REGRESSION ANALYSIS — COMPLETE STEPS")
print("=" * 60)

print("\n[STEP 1] Summary Statistics")
x_bar_f = np.mean(hs_grad)
y_bar_f = np.mean(poverty)
sx_f    = np.std(hs_grad, ddof=1)
sy_f    = np.std(poverty, ddof=1)
R_f     = np.corrcoef(hs_grad, poverty)[0,1]
print(f"  x̄ = {x_bar_f:.2f},  sx = {sx_f:.2f}")
print(f"  ȳ = {y_bar_f:.2f},  sy = {sy_f:.2f}")
print(f"  Correlation R = {R_f:.3f}")

print("\n[STEP 2] Coefficients")
b1_f = (sy_f / sx_f) * R_f
b0_f = y_bar_f - b1_f * x_bar_f
print(f"  Slope:     b₁ = (sy/sx)×R = ({sy_f:.2f}/{sx_f:.2f})×{R_f:.3f} = {b1_f:.4f}")
print(f"  Intercept: b₀ = ȳ - b₁×x̄ = {y_bar_f:.2f} - ({b1_f:.4f})×{x_bar_f:.2f} = {b0_f:.4f}")
print(f"  Model: ŷ = {b0_f:.2f} + ({b1_f:.2f})×x")

print("\n[STEP 3] Model Fit")
y_pred_f = b0_f + b1_f * hs_grad
SS_res_f = np.sum((poverty - y_pred_f)**2)
SS_tot_f = np.sum((poverty - y_bar_f)**2)
R2_f     = 1 - SS_res_f / SS_tot_f
RMSE_f   = np.sqrt(SS_res_f / (len(hs_grad)-2))
print(f"  R² = {R2_f:.4f} → {R2_f*100:.1f}% of variability explained")
print(f"  RMSE = {RMSE_f:.4f} (residual standard error)")

print("\n[STEP 4] Inference (for slope)")
SE_b1_f  = RMSE_f / np.sqrt(np.sum((hs_grad - x_bar_f)**2))
T_f      = b1_f / SE_b1_f
df_f     = len(hs_grad) - 2
p_f      = 2 * (1 - stats.t.cdf(abs(T_f), df=df_f))
t_star_f = stats.t.ppf(0.975, df=df_f)
ci_low   = b1_f - t_star_f * SE_b1_f
ci_high  = b1_f + t_star_f * SE_b1_f
print(f"  SE_b₁ = {SE_b1_f:.4f}")
print(f"  T = {T_f:.4f},  df = {df_f},  p-value = {p_f:.6f}")
print(f"  95% Confidence Interval: ({ci_low:.4f}, {ci_high:.4f})")
print(f"  Result: {'REJECT H₀ ✓' if p_f < 0.05 else 'Fail to reject H₀'}  (p {'<' if p_f < 0.05 else '>='} 0.05)")

print("\n" + "=" * 60)
print("KEY FORMULAS")
print("=" * 60)
formulas = {
    "Slope":              "b₁ = (sy/sx) × R",
    "Intercept":          "b₀ = ȳ - b₁×x̄",
    "Residual":           "eᵢ = yᵢ - ŷᵢ",
    "R²":                 "R² = r²  (square of the correlation)",
    "T statistic":        "T = b₁ / SE_b₁",
    "Degrees of freedom": "df = n - 2",
    "Confidence interval":"b₁ ± t*_(n-2) × SE_b₁",
}
for name, formula in formulas.items():
    print(f"  {name:<25} {formula}")

---
## Practice Questions

Answer the following questions by writing code:

1. Predict the poverty rate for a state with a high school graduation rate of **88%**.
2. If R² = 0.56 in the model from the slides, what proportion of variability is unexplained?
3. Build a simple linear regression model for the dataset below, interpret the coefficients, and check the conditions.

In [ ]:
# Exercise 1: Prediction
hs_grad_exercise = 88
# Write your code here:
# prediction = ...
# print(f"Prediction: {prediction:.2f}")

# Exercise 2: Unexplained variability
R2_given = 0.56
# Write your code here:
# unexplained = ...
# print(f"Unexplained: {unexplained*100:.0f}%")

# Exercise 3: New dataset
np.random.seed(99)
study_hours = np.random.uniform(20, 60, 40)
exam_score  = 30 + 1.2 * study_hours + np.random.normal(0, 8, 40)
exam_score  = np.clip(exam_score, 0, 100)

print("Exercise 3 data:")
print(f"  Study hours: mean={study_hours.mean():.1f}, sd={study_hours.std():.1f}")
print(f"  Exam score:  mean={exam_score.mean():.1f}, sd={exam_score.std():.1f}")
print(f"  Correlation: {np.corrcoef(study_hours, exam_score)[0,1]:.3f}")
print()
print("TODO: Build and interpret a linear regression model for this data.")